# Tidal inundation duration

Estimate mean daily inundation for a DEM tile under a specified accretion, subsidence and sea-level scenario. See README.md for model installation and the worked example.

In [ ]:
# Default example: tile 6, using the scenario settings from the original notebook.
DEM_DIR = "data/dem"
TIDE_MODEL_DIR = "tide_models"
OUTPUT_ROOT = "results"
TIDE_CACHE_DIR = "cache"
GRID_IDS = [6]  # Use range(1, 87) for the full set of study tiles.

BASE_YEAR = 2023
YEAR = 2040
SSP = "SSP2-4.5"
k = 0.05  # Annual fractional decline in sedimentation rate.
S = 0.002  # Annual subsidence in metres.
START_DATE = "2022-01-01"
END_DATE = "2023-12-31 23:00:00"
GRID_RESOLUTION = 0.03  # Target tide grid spacing in degrees.
BUFFER_DISTANCE = 0.03  # Angular distance in degrees from valid DEM pixels.
INTERPOLATION_METHOD = "nearest"
OUTPUT_TIDE_GRID = True
TIDE_GRID_FORMAT = "shp"  # Use "netcdf" for the full hourly series.
PLOT_GRID_COVERAGE = True
USE_TIDE_CACHE = True

## Model functions

In [ ]:
import rasterio
from pyproj import Transformer
import numpy as np
import pandas as pd
from scipy.ndimage import label
import os
from tqdm import tqdm
from scipy.interpolate import RegularGridInterpolator
import time
import geopandas as gpd
from shapely.geometry import Point, box
import xarray as xr
import math
from sklearn.neighbors import KDTree
import matplotlib.pyplot as plt
from eo_tides.model import model_tides

# ================================================
# functions
# ================================================
def calculate_daily_stats(mask, hours_per_day=24):
    """Calculate the average daily inundation duration (hours/day) and the maximum continuous duration"""
    total_days = len(mask) // hours_per_day
    daily_hours = np.zeros(total_days)

    for d in range(total_days):
        daily_slice = slice(d * hours_per_day, (d + 1) * hours_per_day)
        daily_hours[d] = np.sum(mask[daily_slice])

    avg_daily = np.mean(daily_hours)


    labeled, n_components = label(mask)
    if n_components == 0:
        max_consecutive = 0
    else:
        component_sizes = np.bincount(labeled.ravel())[1:]
        max_consecutive = np.max(component_sizes) if len(component_sizes) > 0 else 0

    return avg_daily, max_consecutive

def generate_tide_series(lon, lat,times):
    directory = TIDE_MODEL_DIR

    try:
        tide_df = model_tides(
            x=lon,
            y=lat,
            model=["TPXO10-atlas-v2-nc"],
            time=times,
            directory=directory
        )
        return tide_df['tide_height'].values.astype(np.float32)
    except Exception as e:
        raise RuntimeError(f"Tide prediction failed at ({lon:.4f}, {lat:.4f}).") from e

def save_tide_grid(grid_lons, grid_lats, tide_grid, times, output_dir, format="shp"):
    """Save tidal level"""
    os.makedirs(output_dir, exist_ok=True)

    if format.lower() == "shp":

        points = []
        attributes = []

        max_tide = np.max(tide_grid, axis=0)
        min_tide = np.min(tide_grid, axis=0)
        mean_tide = np.mean(tide_grid, axis=0)
        std_tide = np.std(tide_grid, axis=0)

        for i in range(grid_lons.shape[0]):
            for j in range(grid_lons.shape[1]):
                if np.isnan(mean_tide[i, j]):
                    continue

                point = Point(grid_lons[i, j], grid_lats[i, j])
                points.append(point)
                attributes.append({
                    "grid_id": f"{i}_{j}",
                    "max_tide": max_tide[i, j],
                    "min_tide": min_tide[i, j],
                    "mean_tide": mean_tide[i, j],
                    "std_tide": std_tide[i, j]
                })

        if points:
            gdf = gpd.GeoDataFrame(attributes, geometry=points, crs="EPSG:4326")
            output_path = os.path.join(output_dir, f"tide_grid_points_{GRID_ID}.shp")
            gdf.to_file(output_path)
            print(f"Save as Shapefile: {output_path} (Total: {len(points)})")
        else:
            print("No tide saved")

    elif format.lower() == "netcdf":
        valid_mask = ~np.isnan(np.mean(tide_grid, axis=0))
        valid_lons = grid_lons[valid_mask]
        valid_lats = grid_lats[valid_mask]
        valid_tide = tide_grid[:, valid_mask]

        point_ids = [f"{i}_{j}" for i, j in np.argwhere(valid_mask)]

        ds = xr.Dataset(
            {
                "tide": (["time", "point"], valid_tide),
            },
            coords={
                "time": times,
                "lon": (["point"], valid_lons),
                "lat": (["point"], valid_lats),
                "point_id": (["point"], point_ids)
            }
        )

        ds.attrs["description"] = "tide data"
        ds.attrs["resolution"] = f"{GRID_RESOLUTION}"
        ds.attrs["model"] = "output"
        ds.attrs["valid_points"] = len(valid_lons)

        output_path = os.path.join(output_dir, f"tide_grid_{GRID_ID}.nc")
        ds.to_netcdf(output_path)
        print(f"Tide save as NetCDF: {output_path} (Total: {len(valid_lons)})")

    else:
        print(f"Wrong format: {format}")

def create_geographic_grid(min_lon, max_lon, min_lat, max_lat, resolution):

    num_lon = math.ceil((max_lon - min_lon) / resolution) + 1
    num_lat = math.ceil((max_lat - min_lat) / resolution) + 1


    lons = np.linspace(min_lon, max_lon, num_lon)
    lats = np.linspace(min_lat, max_lat, num_lat)

    lon_grid, lat_grid = np.meshgrid(lons, lats)

    return lon_grid, lat_grid, lons, lats

def filter_grid_by_dem_coverage(grid_lons, grid_lats, dem_lons, dem_lats, buffer_deg=0.05):

    dem_points = np.column_stack((dem_lons.ravel(), dem_lats.ravel()))
    dem_tree = KDTree(dem_points)

    grid_points = np.column_stack((grid_lons.ravel(), grid_lats.ravel()))

    distances, _ = dem_tree.query(grid_points, k=1)
    distances = distances.reshape(grid_lons.shape)

    valid_mask = distances < buffer_deg

    num_valid = np.sum(valid_mask)
    num_total = grid_lons.size
    print(f"Tide grid: retained {num_valid} of {num_total} points ({num_valid/num_total:.1%}).")

    valid_indices = np.argwhere(valid_mask)

    return valid_mask, valid_indices

def plot_grid_coverage(dem_lons, dem_lats, grid_lons, grid_lats, valid_mask, output_dir):
    """Plot the DEM footprint and the tide prediction grid."""
    plt.figure(figsize=(12, 10))

    # Plot the valid DEM pixels.
    plt.scatter(dem_lons, dem_lats, s=1, c='blue', alpha=0.3, label='DEM coverage')

    # Show all candidate tide grid points.
    plt.scatter(grid_lons, grid_lats, s=5, c='gray', alpha=0.3, label='All grids')

    # Highlight grid points retained for tide prediction.
    valid_lons = grid_lons[valid_mask]
    valid_lats = grid_lats[valid_mask]
    plt.scatter(valid_lons, valid_lats, s=10, c='red', alpha=0.7, label='Tide grids')

    # Label the geographic coordinates.
    plt.title('Tide modelling area')
    plt.xlabel('Lon')
    plt.ylabel('Lat')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Save the coverage map.
    output_path = os.path.join(output_dir, f"grid_coverage_{GRID_ID}.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Saved grid coverage map to {output_path}")
    plt.close()

def save_tide_cache(cache_dir, grid_lons, grid_lats, tide_grid, times, valid_indices):
    """Write the hourly tide grid and its coordinates to a NetCDF cache."""
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = os.path.join(cache_dir, f"tide_cache_{GRID_ID}.nc")

    # Store predictions on their original time and spatial axes.
    ds = xr.Dataset(
        {
            "tide": (["time", "y", "x"], tide_grid),
            "valid_mask": (["y", "x"], ~np.isnan(tide_grid[0])),
        },
        coords={
            "time": times,
            "lon": (["y", "x"], grid_lons),
            "lat": (["y", "x"], grid_lats),
        }
    )

    # Record the settings used to generate the tides.
    ds.attrs["dem_path"] = DEM_PATH
    ds.attrs["start_date"] = START_DATE
    ds.attrs["end_date"] = END_DATE
    ds.attrs["grid_resolution"] = GRID_RESOLUTION
    ds.attrs["buffer_distance"] = BUFFER_DISTANCE
    ds.attrs["valid_indices_count"] = len(valid_indices)

    # Write the NetCDF file.
    ds.to_netcdf(cache_file)
    print(f"Saved tide cache to {cache_file}")

    return cache_file

def load_tide_cache(cache_dir):
    """Read an existing tide grid for this DEM tile and time window."""
    cache_file = os.path.join(cache_dir, f"tide_cache_{GRID_ID}.nc")

    if not os.path.exists(cache_file):
        return None, None, None, None, None, None

    print(f"Reading tide cache from {cache_file}")
    ds = xr.open_dataset(cache_file)

    # Read the arrays into memory before closing the file.
    grid_lons = ds["lon"].values
    grid_lats = ds["lat"].values
    tide_grid = ds["tide"].values
    times = ds["time"].values
    valid_mask = ds["valid_mask"].values

    ds.close()

    # Recover the grid indices with valid predictions.
    valid_indices = np.argwhere(valid_mask)

    # Reject a cache built with different settings.
    if (
        ds.attrs["dem_path"] != DEM_PATH or
        ds.attrs["start_date"] != START_DATE or
        ds.attrs["end_date"] != END_DATE or
        ds.attrs["grid_resolution"] != GRID_RESOLUTION or
        ds.attrs["buffer_distance"] != BUFFER_DISTANCE):
        print("Cache settings differ. Recomputing tides.")
        return None, None, None, None, None, None

    print(f"Loaded tide grid {tide_grid.shape} (time, latitude, longitude).")
    return grid_lons, grid_lats, tide_grid, times, valid_indices, valid_mask

def load_dem_data(dem_path):
    """Read the elevation raster and transform its grid coordinates to WGS 84."""
    print("Reading DEM...")
    with rasterio.open(dem_path) as src:
        dem = src.read(1)
        profile = src.profile.copy()
        transform = src.transform
        crs = src.crs
        rows, cols = dem.shape
        nodata = src.nodata
        print(f"DEM dimensions: {rows} rows by {cols} columns. Pixel width: {transform.a:g} CRS units.")

    # Apply the raster transform at integer column and row indices.
    print("Computing DEM coordinates...")
    dem_xs, dem_ys = np.meshgrid(np.arange(cols), np.arange(rows))
    dem_x_coords, dem_y_coords = transform * (dem_xs, dem_ys)

    # Transform coordinates to longitude and latitude.
    transformer_to_geo = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
    dem_lons, dem_lats = transformer_to_geo.transform(dem_x_coords, dem_y_coords)

    # Identify pixels with valid elevations.
    valid_dem_mask = (dem != nodata) & (~np.isnan(dem))

    return {
        "dem": dem,
        "profile": profile,
        "transform": transform,
        "crs": crs,
        "rows": rows,
        "cols": cols,
        "nodata": nodata,
        "dem_lons": dem_lons,
        "dem_lats": dem_lats,
        "valid_dem_mask": valid_dem_mask
        }

def decayed_sum(k, T):
    total = 0.0
    for t in range(1, T+1):  # Start the accumulation at year one.
        term =(1 - k) ** t  # Fraction of the initial annual accretion rate in year t.
        total =total+term
    return total

# ================================================
# Main workflow
# ================================================
def main():
    start_time = time.time()
    print(f"Started at {time.strftime('%Y-%m-%d %H:%M:%S')}")

    # Create the output directory.
    os.makedirs(OUTPUT_DIR, exist_ok=True)


    # Reuse matching hourly predictions if available.
    tide_data_loaded = False
    if USE_TIDE_CACHE:
        grid_lons, grid_lats, tide_grid, times, valid_indices, valid_mask = load_tide_cache(
            TIDE_CACHE_DIR
        )
        if grid_lons is not None:
            tide_data_loaded = True
            grid_rows, grid_cols = grid_lons.shape
            total_hours = len(times)
            total_days = total_hours // 24
    # Load the DEM for both newly computed and cached tides.
    dem_data = load_dem_data(DEM_PATH)
    dem = dem_data["dem"]
    profile = dem_data["profile"]
    rows = dem_data["rows"]
    cols = dem_data["cols"]
    nodata = dem_data["nodata"]
    dem_lons = dem_data["dem_lons"]
    dem_lats = dem_data["dem_lats"]
    valid_dem_mask = dem_data["valid_dem_mask"]
    # Compute the prediction grid on the first run.
    if not tide_data_loaded:
        valid_dem_lons = dem_lons[valid_dem_mask]
        valid_dem_lats = dem_lats[valid_dem_mask]

        min_lon, max_lon = np.min(valid_dem_lons), np.max(valid_dem_lons)
        min_lat, max_lat = np.min(valid_dem_lats), np.max(valid_dem_lats)

        # Extend the bounding box by the specified angular buffer.
        buffer = BUFFER_DISTANCE
        min_lon -= buffer
        min_lat -= buffer
        max_lon += buffer
        max_lat += buffer

        print(f"Buffered extent: {min_lon:.4f} to {max_lon:.4f} longitude, {min_lat:.4f} to {max_lat:.4f} latitude.")

        # Create a regular geographic grid.
        print("Creating tide grid...")
        grid_lons, grid_lats, lon_coords, lat_coords = create_geographic_grid(
            min_lon, max_lon, min_lat, max_lat, GRID_RESOLUTION
        )

        grid_rows, grid_cols = grid_lons.shape
        print(f"Tide grid: {grid_rows} rows by {grid_cols} columns ({grid_rows * grid_cols} points).")

        # Keep points within the buffer distance of a valid DEM pixel.
        print("Filtering tide grid points by distance to the DEM...")
        valid_mask, valid_indices = filter_grid_by_dem_coverage(
            grid_lons, grid_lats, valid_dem_lons, valid_dem_lats, BUFFER_DISTANCE
        )
        # Plot the retained grid points.
        if PLOT_GRID_COVERAGE:
            plot_grid_coverage(valid_dem_lons, valid_dem_lats, grid_lons, grid_lats, valid_mask, OUTPUT_DIR)

        # Generate timestamps at hourly intervals, including both endpoints.
        print("Generating hourly timestamps...")
        times = pd.date_range(start=START_DATE, end=END_DATE, freq="h")
        total_hours = len(times)
        total_days = total_hours // 24
        print(f"Time window: {START_DATE} to {END_DATE} ({total_hours} hours, {total_days} complete days).")

        # Predict the tide series for each retained grid point.
        print("Predicting hourly tides...")
        tide_grid = np.full((total_hours, grid_rows, grid_cols), np.nan, dtype=np.float32)

        # Leave excluded points as NaN.
        for idx in tqdm(valid_indices, desc="Predicting tides", unit="point"):
            i, j = idx
            tide_grid[:, i, j] = generate_tide_series(grid_lons[i, j], grid_lats[i, j], times)

        # Cache the predictions for repeat runs.
        if USE_TIDE_CACHE:
            cache_file = save_tide_cache(
                TIDE_CACHE_DIR,
                grid_lons, grid_lats, tide_grid, times, valid_indices
            )

    # Export the newly predicted grid if requested.
    if OUTPUT_TIDE_GRID and not tide_data_loaded:
        print("Exporting tide grid...")
        save_tide_grid(grid_lons, grid_lats, tide_grid, times, OUTPUT_DIR, TIDE_GRID_FORMAT)

    # Build one spatial interpolator per hour.
    print("Building tide interpolators...")
    interpolators = []

    # Reconstruct the coordinate axes from the cached mesh.
    if tide_data_loaded:
        lon_coords = grid_lons[0, :]
        lat_coords = grid_lats[:, 0]

    for t_idx in tqdm(range(total_hours), desc="Building interpolators", unit="hour"):
        # Interpolate this hour's tide heights.
        interpolator = RegularGridInterpolator(
            (lat_coords, lon_coords),  # Array axes are latitude first, longitude second.
            tide_grid[t_idx, :, :],
            method=INTERPOLATION_METHOD,
            bounds_error=False,
            fill_value=np.nan
        )
        interpolators.append(interpolator)

    # Allocate output statistics.
    stats = {
        "avg_daily_flood": np.full(dem.shape, np.nan, dtype=np.float32),
        # "max_consecutive_flood": np.zeros(dem.shape, dtype=np.int32),
        # "flood_days": np.zeros(dem.shape, dtype=np.int32),
    }

    # Arrange DEM coordinates in the order expected by the interpolator.
    dem_points = np.column_stack((dem_lats.ravel(), dem_lons.ravel()))

    # Evaluate inundation at each hourly timestamp.
    print("Calculating hourly inundation...")
    flood_mask = np.zeros((total_hours, rows, cols), dtype=bool)

    delta_h=A[GRID_ID-1]*decayed_sum(k=k, T=T)*0.01
    print(f"SSP={SSP}; YEAR={YEAR}; GRID={GRID_ID}; A={A[GRID_ID-1]}; delta_h={delta_h}; delta_slr={delta_slr}")

    dem=dem+delta_h-S*T-delta_slr


    for t_idx in tqdm(range(total_hours), desc="Evaluating inundation", unit="hour"):
        # Select the interpolator for this hour.
        interp = interpolators[t_idx]

        # Estimate tide height at each DEM pixel.
        tide_values = interp(dem_points).reshape(rows, cols)


        # Compare tide height and adjusted elevation at valid pixels.
        if not np.all(np.isfinite(tide_values[valid_dem_mask])):
            raise ValueError("Missing tide over valid DEM pixels. Check the tide model, grid coverage, interpolation and cache.")


        flood_mask[t_idx] = (tide_values > dem) & valid_dem_mask

    # Aggregate the hourly series into daily statistics.
    print("Calculating daily statistics...")
    for i in tqdm(range(rows), desc="Summarising pixels", unit="row"):
        for j in range(cols):
            if not valid_dem_mask[i, j]:
                continue

            # Extract the hourly series at this pixel.
            pixel_flood = flood_mask[:, i, j]

            # Count flooded hours per day and the longest continuous flood.
            avg_daily, max_cons = calculate_daily_stats(pixel_flood)

            # Count days containing at least one flooded hour.
            flood_days = 0
            for d in range(total_days):
                daily_slice = slice(d * 24, (d + 1) * 24)
                if np.any(pixel_flood[daily_slice]):
                    flood_days += 1

            # Store the selected output statistic.
            stats["avg_daily_flood"][i, j] = avg_daily
            # stats["max_consecutive_flood"][i, j] = max_cons
            # stats["flood_days"][i, j] = flood_days

    # Write the statistic as a GeoTIFF.
    print("Writing inundation raster...")
    profile.update(dtype=rasterio.float32, nodata=-9999)
    for key in stats:
        output_path = os.path.join(OUTPUT_DIR, f"{key}_{GRID_ID}_{YEAR}.tif")
        with rasterio.open(output_path, "w", **profile) as dst:
            # Report mean daily inundation to one decimal place.
            if key == "avg_daily_flood":
                data = np.round(stats[key], 1)
            else:
                data = stats[key]
            dst.write(np.where(valid_dem_mask, data, -9999).astype(np.float32), 1)
        print(f"Saved {output_path}")

    # Report elapsed time and grid dimensions.
    end_time = time.time()
    elapsed = end_time - start_time
    hours, rem = divmod(elapsed, 3600)
    minutes, seconds = divmod(rem, 60)
    print(f"Finished in {int(hours):02d}h {int(minutes):02d}m {seconds:.1f}s.")

    if not tide_data_loaded:
        print(f"DEM: {rows} by {cols} ({rows*cols:,} pixels).")
        print(f"Time series: {total_hours} hours ({total_days} days).")
        print(f"Tide grid: {grid_rows} by {grid_cols} ({grid_rows*grid_cols:,} points).")
        print(f"Retained tide grid points: {len(valid_indices):,}")
    del flood_mask,stats,dem,valid_dem_mask,dem_data,profile

## Scenario inputs

Decadal sea-level projections in metres and initial annual sedimentation rates in centimetres per year. The first sedimentation rate corresponds to tile 1.

In [ ]:
data = {
'SSP1-2.6': [0.049, 0.092, 0.136, 0.189, 0.232, 0.29, 0.338, 0.386, 0.436, 0.497, 0.545, 0.591, 0.637, 0.681],
'SSP2-4.5': [0.049, 0.093, 0.142, 0.204, 0.262, 0.33, 0.403, 0.476, 0.556, 0.628, 0.702, 0.775, 0.848, 0.919],
'SSP5-8.5': [0.05, 0.098, 0.157, 0.232, 0.31, 0.404, 0.509, 0.633, 0.766, 0.856, 0.98, 1.1, 1.214, 1.322]
}
years = np.arange(2020, 2160, 10)
df = pd.DataFrame(data, index=years)
A=[1.96,1.96,1.96,1.96,0.46,0.46,0.46,0.46,0.46,2.24,
2.24,2.24,2.24,2.0,2.0,2.0,2.0,2.0,2.0,2.0,
2.0,2.0,2.33,2.33,2.33,0.85,0.85,0.85,0.85,0.85,
1.14,1.14,2.3,2.3,2.3,2.3,2.2,2.2,2.2,2.2,
2.2,4.6,4.6,4.6,2.58,2.58,2.58,1.63,1.63,1.63,
1.63,3.01,3.01,3.01,1.92,1.92,0.46,0.46,0.4,0.4,
0.86,0.86,0.51,0.51,3.5,1.24,1.24,1.24,1.24,1.24,
1.15,1.15,1.15,1.15,1.15,0.87,0.87,0.79,0.79,1.83,
1.83,1.83,1.83,0.46,0.46,0.46]

## Run the selected tiles

In [ ]:
T = YEAR - BASE_YEAR
delta_slr = 0.0 if YEAR == BASE_YEAR else df.loc[YEAR, SSP] - 0.049
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, f"{SSP}_k{k:g}_S{S:g}", str(YEAR))
for GRID_ID in GRID_IDS:
    if not 1 <= GRID_ID <= len(A):
        raise ValueError(f"Use tile IDs from 1 to {len(A)}.")
    DEM_PATH = os.path.join(DEM_DIR, f"{GRID_ID}_dem.tif")
    main()